## Tranining Model
In this notebook, we will train and experiment with different machine learning, AI, and Statistical methods for classifying anomalous and normal packets. The result from this training will be used as weight for the application demonstration, which can be used in real-time for analyzing anomalous packets in the network through python scapy + web application under `/app` folder.

In [ ]:
!pip install -r requirements.txt

In [2]:
from scipy.io import arff
import pandas as pd

# to avoid base 10 exponents
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

arff_file = arff.loadarff("dataset/KDDTrain+.arff")
train_dataset_df = pd.DataFrame(arff_file[0])

# decode the bytes loaded from scipy (default behavior)
categorical_cols = train_dataset_df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    try:
        train_dataset_df[col] = train_dataset_df[col].str.decode('utf-8')

    # exception triggers when the column is already a string
    except AttributeError:
        continue

# normalize the integer columns from float64 -> integer
numeric_real_cols = [
    "duration",
    "src_bytes",
    "dst_bytes",
    "wrong_fragment",
    "urgent",
    "hot",
    "num_failed_logins",
    "num_compromised",
    "root_shell",
    "su_attempted",
    "num_root",
    "num_file_creations",
    "num_shells",
    "num_access_files",
    "num_outbound_cmds",
    "count",
    "srv_count",
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "srv_diff_host_rate",
    "dst_host_count",
    "dst_host_srv_count",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate",
    "dst_host_srv_serror_rate",
    "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
    "land",
    "logged_in",
    "is_host_login",
    "is_guest_login"
]

for numeric_columns in numeric_real_cols:
    train_dataset_df[numeric_columns] = pd.to_numeric(train_dataset_df[numeric_columns])


In [3]:
arff_file = arff.loadarff("dataset/KDDTest+.arff")
test_dataset_df = pd.DataFrame(arff_file[0])

for col in categorical_cols:
    try:
        test_dataset_df[col] = test_dataset_df[col].str.decode('utf-8')

    # exception triggers when the column is already a string
    except AttributeError:
        continue


In [4]:
from sklearn.preprocessing import LabelEncoder

# training dataset
x_train = train_dataset_df.drop(columns=['class'])
y_train = train_dataset_df['class']

# testing dataset
x_test = test_dataset_df.drop(columns=['class'])
y_test = test_dataset_df['class']


# class preprpocessing from string label to numerical
for col in x_train.select_dtypes(include=["object", "str"]).columns:
    x_train[col] = LabelEncoder().fit_transform(x_train[col].astype(str))


for col in x_test.select_dtypes(include=["object", "str"]).columns:
    x_test[col] = LabelEncoder().fit_transform(x_test[col].astype(str))


### Straightforward Approach
The following training below uses a straightforward approach or simply training the data directly to the model, the purpose of this is for us to have a high-level overview which model could potentially work very well for the dataset

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.fit_transform(x_test)

# logistic regression training
logistic_regression_model = LogisticRegression(max_iter=500, random_state=67)
logistic_regression_model.fit(x_scaled, y_train)

y_predict = logistic_regression_model.predict(x_test_scaled)

# metrics
cls_report = classification_report(y_test, y_predict)

print("==== Logistic Regression Report ====")
print()
print(cls_report)

==== Logistic Regression Report ====

              precision    recall  f1-score   support

     anomaly       0.91      0.68      0.78     12833
      normal       0.69      0.91      0.78      9711

    accuracy                           0.78     22544
   macro avg       0.80      0.80      0.78     22544
weighted avg       0.82      0.78      0.78     22544

